# Práctica — Clase 6. Control sin modelo: SARSA y Q-learning

Esta práctica te toca a ti. Trabajas de forma autónoma con el mismo entorno **Cliff Walking** (el acantilado) que construiste en clase, y resuelves dos ejercicios sobre la actualización de **SARSA** (on-policy) y de **Q-learning** (off-policy).

**Cómo trabajar con este notebook:**
1. Ejecuta primero la sección "Preparación", que reconstruye el entorno de la clase
2. En cada ejercicio, completa las líneas marcadas con `# TODO`
3. Ejecuta tu solución y compara el resultado con lo que se describe en el enunciado
4. Solo después, despliega la solución oculta para comprobar tu respuesta

> **No mires la solución antes de intentarlo.** El valor de este ejercicio está en pelearte un poco con la diferencia entre "el Q de la acción que sigo" y "el máximo Q posible", no en leer la respuesta.


## Preparación: recupera el entorno de la clase

Esta celda reproduce tal cual el entorno del acantilado y las funciones auxiliares (`paso`, `politica_epsilon_greedy`, `extraer_ruta_greedy`, `media_movil`, `dibujar_ruta`) que construiste en el notebook de teoría. Los ejercicios de hoy reutilizan estos mismos nombres, así que no los cambies.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FILAS, COLUMNAS = 4, 12
INICIO = (3, 0)
META = (3, 11)
ACANTILADO = [(3, c) for c in range(1, 11)]
NUM_ACCIONES = 4


def paso(estado, accion):
    """Aplica una acción y devuelve (nuevo_estado, recompensa, terminado)."""
    fila, col = estado
    if accion == 0:
        fila = max(fila - 1, 0)
    elif accion == 1:
        fila = min(fila + 1, FILAS - 1)
    elif accion == 2:
        col = max(col - 1, 0)
    elif accion == 3:
        col = min(col + 1, COLUMNAS - 1)

    nuevo_estado = (fila, col)

    if nuevo_estado in ACANTILADO:
        return INICIO, -100, False
    if nuevo_estado == META:
        return nuevo_estado, -1, True
    return nuevo_estado, -1, False


def politica_epsilon_greedy(Q, estado, epsilon):
    """Con probabilidad epsilon explora; si no, elige la mejor acción conocida."""
    if np.random.rand() < epsilon:
        return np.random.randint(NUM_ACCIONES)
    valores = Q[estado]
    return np.random.choice(np.flatnonzero(valores == valores.max()))


def extraer_ruta_greedy(Q, max_pasos=50):
    """Sigue la política puramente codiciosa (sin exploración) desde el inicio."""
    estado = INICIO
    ruta = [estado]
    for _ in range(max_pasos):
        if estado == META:
            break
        accion = np.argmax(Q[estado])
        estado, _, terminado = paso(estado, accion)
        ruta.append(estado)
        if terminado:
            break
    return ruta


def media_movil(valores, ventana=20):
    return np.convolve(valores, np.ones(ventana) / ventana, mode="valid")


def dibujar_ruta(ax, ruta, titulo):
    cuadricula = np.zeros((FILAS, COLUMNAS))
    for (f, c) in ACANTILADO:
        cuadricula[f, c] = -1
    ax.imshow(cuadricula, cmap="Pastel1")
    filas_ruta = [p[0] for p in ruta]
    columnas_ruta = [p[1] for p in ruta]
    ax.plot(columnas_ruta, filas_ruta, marker="o", color="tab:blue")
    ax.scatter(*INICIO[::-1], color="green", s=100, label="Inicio", zorder=5)
    ax.scatter(*META[::-1], color="red", s=100, label="Meta", zorder=5)
    ax.set_title(titulo)
    ax.invert_yaxis()
    ax.legend(loc="upper center")


---

## Ejercicio 1: completa la actualización de SARSA (on-policy)

SARSA elige la acción siguiente **antes** de actualizar, con la misma política ε-greedy que usa para actuar, y usa el Q de esa acción concreta como objetivo. Completa la función `entrenar_sarsa_ejercicio` con los tres huecos que faltan dentro del bucle:

```
Q(S,A) se acerca un poco a: recompensa recibida + valor descontado de Q en el estado
                             siguiente, para la acción que la política eligió de verdad
```

**Pistas:**
- El primer hueco elige `siguiente_accion` con `politica_epsilon_greedy(Q, siguiente_estado, epsilon)`, exactamente igual que se elige `accion` al principio de cada episodio
- El segundo hueco calcula `objetivo`, combinando `recompensa`, `gamma` y `Q[siguiente_estado][siguiente_accion]`
- El tercer hueco actualiza `Q[estado][accion]` acercándolo a `objetivo` con la tasa de aprendizaje `alpha`: es la misma regla TD que ya conoces de la clase anterior, `Q[estado][accion] += alpha * (objetivo - Q[estado][accion])`
- Con los hiperparámetros por defecto (500 episodios, α=0.5, γ=1.0, ε=0.1) y semilla 0, la recompensa media de los últimos 50 episodios debería rondar los −32


In [ ]:
def entrenar_sarsa_ejercicio(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """SARSA: actualización on-policy. Completa las tres líneas marcadas con TODO."""
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        accion = politica_epsilon_greedy(Q, estado, epsilon)
        recompensa_total = 0
        terminado = False

        while not terminado:
            siguiente_estado, recompensa, terminado = paso(estado, accion)

            # TODO: elige la siguiente acción con la política epsilon-greedy.
            # SARSA necesita saber AHORA qué acción va a tomar después.
            siguiente_accion = None

            # TODO: calcula el objetivo on-policy: recompensa + gamma * Q de
            # la acción que SÍ vas a tomar a continuación (siguiente_accion)
            objetivo = None

            # TODO: actualiza Q[estado][accion] acercándolo a 'objetivo'
            # con la tasa de aprendizaje alpha
            Q[estado][accion] += 0

            estado, accion = siguiente_estado, siguiente_accion
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


np.random.seed(0)
Q_sarsa_ejercicio, recompensas_sarsa_ejercicio = entrenar_sarsa_ejercicio()

print(f"Recompensa media de los últimos 50 episodios (SARSA): {np.mean(recompensas_sarsa_ejercicio[-50:]):.1f}")
print("Ruta final aprendida (SARSA):", extraer_ruta_greedy(Q_sarsa_ejercicio))


<details>
<summary>💡 Ver solución</summary>

```python
def entrenar_sarsa_ejercicio(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        accion = politica_epsilon_greedy(Q, estado, epsilon)
        recompensa_total = 0
        terminado = False

        while not terminado:
            siguiente_estado, recompensa, terminado = paso(estado, accion)

            # Elegimos ya la siguiente acción con la política epsilon-greedy:
            # es la acción que de verdad vamos a ejecutar en el próximo paso
            siguiente_accion = politica_epsilon_greedy(Q, siguiente_estado, epsilon)

            # Objetivo on-policy: usamos el Q de esa acción concreta,
            # no el máximo entre todas las acciones posibles
            objetivo = recompensa + gamma * Q[siguiente_estado][siguiente_accion]

            # Regla TD de siempre: acercamos Q[estado][accion] al objetivo
            Q[estado][accion] += alpha * (objetivo - Q[estado][accion])

            estado, accion = siguiente_estado, siguiente_accion
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


np.random.seed(0)
Q_sarsa_ejercicio, recompensas_sarsa_ejercicio = entrenar_sarsa_ejercicio()

print(f"Recompensa media de los últimos 50 episodios (SARSA): {np.mean(recompensas_sarsa_ejercicio[-50:]):.1f}")
print("Ruta final aprendida (SARSA):", extraer_ruta_greedy(Q_sarsa_ejercicio))

# Salida esperada aproximada (los valores exactos pueden variar un poco
# según la versión de numpy, pero deberían acercarse a esto):
# Recompensa media de los últimos 50 episodios (SARSA): -32.5
# Ruta final aprendida (SARSA): [(3, 0), (2, 0), (1, 0), (1, 1), (0, 1), ...,
#                                 (0, 11), (1, 11), (2, 11), (3, 11)]
#
# Fíjate en la ruta: se aleja del acantilado subiendo hacia la fila 0
# (la más alejada del borde), justo el comportamiento prudente que
# esperabas de un algoritmo on-policy.
```
</details>


---

## Ejercicio 2: completa la actualización de Q-learning (off-policy) y compara la estabilización

Q-learning cambia un solo detalle frente a SARSA: en vez del Q de la acción que realmente tomarás después, usa el **máximo** Q posible entre todas las acciones del estado siguiente. Completa `entrenar_q_learning_ejercicio` y, después, usa el código ya dado para comparar cuántos episodios tarda cada algoritmo en estabilizar su recompensa.

```
Q(S,A) se acerca un poco a: recompensa recibida + valor descontado del mejor Q
                             posible en el estado siguiente (el máximo sobre todas las acciones)
```

**Pistas:**
- El primer hueco calcula `mejor_valor_siguiente` con `Q[siguiente_estado].max()`, sin elegir ninguna acción concreta
- El segundo hueco calcula `objetivo`, combinando `recompensa`, `gamma` y `mejor_valor_siguiente`
- El tercer hueco es idéntico al de SARSA: `Q[estado][accion] += alpha * (objetivo - Q[estado][accion])`
- La función `episodio_de_estabilizacion` ya está completa: busca el primer episodio en el que la media móvil de la recompensa se acerca a menos de un margen de su propio valor final. Es una forma aproximada de detectar cuándo la curva deja de mejorar y entra en su meseta
- Ejecuta esta celda **después** de la del ejercicio 1, sin volver a ejecutar esa celda: así reutilizas `recompensas_sarsa_ejercicio` para la comparación final


In [ ]:
def entrenar_q_learning_ejercicio(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """Q-learning: actualización off-policy con el máximo. Completa los tres TODO."""
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        recompensa_total = 0
        terminado = False

        while not terminado:
            accion = politica_epsilon_greedy(Q, estado, epsilon)
            siguiente_estado, recompensa, terminado = paso(estado, accion)

            # TODO: calcula el mejor Q posible en el estado siguiente,
            # sin importar qué acción se elija después en la práctica
            mejor_valor_siguiente = None

            # TODO: calcula el objetivo off-policy: recompensa + gamma * mejor_valor_siguiente
            objetivo = None

            # TODO: actualiza Q[estado][accion] acercándolo a 'objetivo'
            # con la tasa de aprendizaje alpha (la misma regla TD de siempre)
            Q[estado][accion] += 0

            estado = siguiente_estado
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


def episodio_de_estabilizacion(recompensas, ventana=20, margen=10.0):
    """Heurística sencilla: primer episodio en el que la media móvil
    (ventana de 20 episodios) queda a menos de 'margen' de su propio
    valor final. Marca, de forma aproximada, cuándo la curva deja de
    mejorar con claridad y entra en su meseta."""
    mm = media_movil(recompensas, ventana)
    objetivo = mm[-1]
    dentro_del_margen = np.abs(mm - objetivo) < margen
    primer_indice = int(np.argmax(dentro_del_margen))
    return primer_indice + ventana


Q_qlearning_ejercicio, recompensas_qlearning_ejercicio = entrenar_q_learning_ejercicio()

print(f"Recompensa media de los últimos 50 episodios (Q-learning): {np.mean(recompensas_qlearning_ejercicio[-50:]):.1f}")

ep_sarsa = episodio_de_estabilizacion(recompensas_sarsa_ejercicio)
ep_qlearning = episodio_de_estabilizacion(recompensas_qlearning_ejercicio)

print(f"\nSARSA se estabiliza aproximadamente en el episodio {ep_sarsa}")
print(f"Q-learning se estabiliza aproximadamente en el episodio {ep_qlearning}")

plt.figure(figsize=(9, 5))
plt.plot(media_movil(recompensas_sarsa_ejercicio), label="SARSA (on-policy)")
plt.plot(media_movil(recompensas_qlearning_ejercicio), label="Q-learning (off-policy)")
plt.axvline(ep_sarsa, color="tab:blue", linestyle="--", alpha=0.5)
plt.axvline(ep_qlearning, color="tab:orange", linestyle="--", alpha=0.5)
plt.xlabel("Episodio")
plt.ylabel("Recompensa por episodio (media móvil)")
plt.title("Comparación de estabilización: SARSA vs Q-learning")
plt.legend()
plt.tight_layout()
plt.show()


<details>
<summary>💡 Ver solución</summary>

```python
def entrenar_q_learning_ejercicio(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        recompensa_total = 0
        terminado = False

        while not terminado:
            accion = politica_epsilon_greedy(Q, estado, epsilon)
            siguiente_estado, recompensa, terminado = paso(estado, accion)

            # Máximo Q del estado siguiente: no depende de qué acción
            # se elija después en la práctica, esa es la clave off-policy
            mejor_valor_siguiente = Q[siguiente_estado].max()

            # Objetivo off-policy: recompensa + valor descontado del mejor
            # Q posible, asumiendo que a partir de ahora actuarías de forma óptima
            objetivo = recompensa + gamma * mejor_valor_siguiente

            # Misma regla TD que en SARSA
            Q[estado][accion] += alpha * (objetivo - Q[estado][accion])

            estado = siguiente_estado
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


def episodio_de_estabilizacion(recompensas, ventana=20, margen=10.0):
    mm = media_movil(recompensas, ventana)
    objetivo = mm[-1]
    dentro_del_margen = np.abs(mm - objetivo) < margen
    primer_indice = int(np.argmax(dentro_del_margen))
    return primer_indice + ventana


Q_qlearning_ejercicio, recompensas_qlearning_ejercicio = entrenar_q_learning_ejercicio()

print(f"Recompensa media de los últimos 50 episodios (Q-learning): {np.mean(recompensas_qlearning_ejercicio[-50:]):.1f}")

ep_sarsa = episodio_de_estabilizacion(recompensas_sarsa_ejercicio)
ep_qlearning = episodio_de_estabilizacion(recompensas_qlearning_ejercicio)

print(f"\nSARSA se estabiliza aproximadamente en el episodio {ep_sarsa}")
print(f"Q-learning se estabiliza aproximadamente en el episodio {ep_qlearning}")

# Salida esperada aproximada, con semilla 0 y ejecutando el ejercicio 1
# justo antes de este (mismo flujo de números aleatorios):
# Recompensa media de los últimos 50 episodios (Q-learning): -38.0
# SARSA se estabiliza aproximadamente en el episodio 75
# Q-learning se estabiliza aproximadamente en el episodio 84
```

**Por qué el número es solo aproximado:** aquí `epsilon` se mantiene fijo en 0.1 durante todo el entrenamiento —no decae, así que no cumple del todo la condición GLIE que viste en el resumen de la clase—. Eso significa que ambos agentes siguen explorando, y por tanto siguen cayendo alguna vez por el acantilado, hasta el último episodio. La "meseta" nunca es perfectamente plana: es una franja ruidosa alrededor de un valor medio. El número de `episodio_de_estabilizacion` es una foto aproximada de cuándo esa franja empieza; para una lectura más completa, mira siempre también la forma de la curva en el gráfico, no solo el número aislado.
</details>


---

## Para seguir practicando

Si te ha sobrado tiempo, vuelve a las funciones completas y prueba estas variantes del capítulo:

1. **Cambia epsilon.** Sube `epsilon` a 0.3 en las dos funciones de entrenamiento y vuelve a ejecutar. Observa si la brecha entre la recompensa de SARSA y la de Q-learning durante el entrenamiento se agranda o se reduce
2. **Cambia la penalización del acantilado.** En la función `paso`, sustituye el −100 por, por ejemplo, −10 o −500, y compara cómo cambian las dos rutas finales. Anota tus observaciones: ¿con una penalización muy baja las dos rutas terminan pareciéndose?

**Siguiente parada:** en la Clase 7 vas a sustituir la tabla Q por una red neuronal, porque una tabla explícita deja de ser viable en cuanto el número de estados se dispara. Ahí nacen las redes Q profundas (DQN, de *Deep Q-Networks*). ¡Guarda este notebook, la lógica de la actualización off-policy que acabas de programar reaparece prácticamente igual!
